# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

repo_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)
os.chdir(repo_root)

df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Rows:", len(df), " | Base rate (declining):", round(df["is_declining_label"].mean(), 3))

Rows: 30000  | Base rate (declining): 0.542



My lane is CTR / Engagement Opportunity Scoring — a ranking problem ("which pages should be
reviewed first"), evaluated with Precision@K on classifier probability scores.

Method progression:
- **Logistic Regression first.** It is interpretable (coefficients show direction and rough
  magnitude of each signal), and per the training-honest-models skill, a yes/no label from an
  observed outcome should start with a readable model before moving to something stronger.
- **Random Forest second.** It can capture non-linear interactions between visibility,
  freshness, and position signals that a linear model cannot express, and it is the model the
  reference pipeline (scripts/03_train_model.py) selects as best on this same dataset.

I am not skipping straight to Random Forest: added complexity only earns its place if the
comparison table in Section 3 shows it actually beats both the baseline and Logistic Regression
on the same split and metric. A simpler model that wins honestly is preferable to a complex one
that wins on inflated numbers.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

print("Train rows:", len(train_idx), " | Test rows:", len(test_idx))
print("Train clients:", client_series.iloc[train_idx].nunique(), " | Test clients:", client_series.iloc[test_idx].nunique())
print("Train declining rate:", round(df["is_declining_label"].iloc[train_idx].mean(), 3))
print("Test declining rate:", round(df["is_declining_label"].iloc[test_idx].mean(), 3))

Train rows: 27675  | Test rows: 2325
Train clients: 26  | Test clients: 6
Train declining rate: 0.555
Test declining rate: 0.391



I use a **client-holdout split**, not a random row split. `client_id` repeats across many rows
(32 clients over 30,000 pages), and pages belonging to the same client likely share hidden
characteristics — same site, same content strategy, similar baseline traffic patterns. A random
row split would let the model partly memorize client-specific quirks instead of learning
generalizable CTR/engagement signals, inflating Precision@K in a way that would not hold on a
client the model has never seen.

I hold out ~20% of **clients** entirely (not 20% of rows) — no client_id present in the test set
appears in training. This mirrors the `client_holdout` strategy in `scripts/03_train_model.py`,
so my Week-5 evaluation stays comparable to the reference pipeline's design.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# --- Rebuild the Week-4 baseline formula so the comparison is apples-to-apples ---
def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(method="average", pct=True).fillna(0)

def normalize(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]

df["baseline_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

# --- Build the feature matrix ---
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "days_with_impressions",
    "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in categorical_features:
    df[col] = df[col].fillna("unknown").astype(str)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
numeric_features += ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d"]

categorical_encoded = pd.get_dummies(df[categorical_features], prefix=categorical_features, dtype=float)
feature_frame = pd.concat(
    [df[numeric_features].reset_index(drop=True), categorical_encoded.reset_index(drop=True)],
    axis=1,
)

X_train, X_test = feature_frame.iloc[train_idx], feature_frame.iloc[test_idx]
y_train, y_test = df["is_declining_label"].iloc[train_idx], df["is_declining_label"].iloc[test_idx]

print("Feature count:", feature_frame.shape[1])

# --- Precision@K helper (same definition as Week 4) ---
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[:k]]
    return float(top.mean())

# --- Evaluate baseline, Logistic Regression, Random Forest on the SAME test split ---
results = {}

baseline_test_scores = df["baseline_score"].iloc[test_idx].to_numpy()
results["baseline"] = {
    "precision_at_20": precision_at_k(y_test, baseline_test_scores, 20),
    "precision_at_50": precision_at_k(y_test, baseline_test_scores, 50),
    "roc_auc": roc_auc_score(y_test, baseline_test_scores),
    "average_precision": average_precision_score(y_test, baseline_test_scores),
}

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
results["logistic_regression"] = {
    "precision_at_20": precision_at_k(y_test, logreg_scores, 20),
    "precision_at_50": precision_at_k(y_test, logreg_scores, 50),
    "roc_auc": roc_auc_score(y_test, logreg_scores),
    "average_precision": average_precision_score(y_test, logreg_scores),
}

rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
results["random_forest"] = {
    "precision_at_20": precision_at_k(y_test, rf_scores, 20),
    "precision_at_50": precision_at_k(y_test, rf_scores, 50),
    "roc_auc": roc_auc_score(y_test, rf_scores),
    "average_precision": average_precision_score(y_test, rf_scores),
}

comparison_table = pd.DataFrame(results).T.round(3)
print("Base rate on test set:", round(y_test.mean(), 3))
comparison_table

Feature count: 54
Base rate on test set: 0.391


,precision_at_20,precision_at_50,roc_auc,average_precision
baseline,0.15,0.22,0.628,0.468
logistic_regression,0.35,0.32,0.700,0.518
random_forest,0.85,0.74,0.752,0.627


## Train + compare vs my baseline

Table above: baseline vs Logistic Regression vs Random Forest, all evaluated on the SAME
held-out clients and the SAME metrics. The base rate is printed next to it, since a high
Precision@K only means something relative to that base rate.

- Baseline: Precision@20 = 0.15, Precision@50 = 0.22, ROC AUC = 0.628, Average precision = 0.468
- Logistic Regression: Precision@20 = 0.35, Precision@50 = 0.32, ROC AUC = 0.700, Average precision = 0.518
- Random Forest: Precision@20 = 0.85, Precision@50 = 0.74, ROC AUC = 0.752, Average precision = 0.627

Test base rate: 0.391 (declining rate on the 6 held-out clients).

Random Forest wins on every single metric here, clearly and consistently — there is no
metric where the baseline or Logistic Regression comes out ahead. At Precision@20, Random
Forest is roughly 5.7x the baseline (0.85 vs 0.15) and more than double Logistic Regression
(0.85 vs 0.35). At Precision@50 the gap narrows a bit (0.74 vs 0.22 baseline, vs 0.32 for
Logistic Regression) but Random Forest still leads by a wide margin.

One caveat before treating this as a settled result: the client-holdout split leaves only
6 clients in the test set, and their declining rate (0.391) differs noticeably from the
training clients' rate (0.555). With so few test clients, a large part of this gap could be
driven by which specific clients landed in the test fold, not purely by model skill. This
result should be read as a strong directional signal on this split, not as a number that
would necessarily reproduce exactly on a different random seed's client assignment.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# --- Feature importance from Random Forest ---
importance = pd.Series(rf.feature_importances_, index=feature_frame.columns).sort_values(ascending=False)
importance.head(10)

# --- Where is the model most wrong? ---
test_frame = df.iloc[test_idx].copy()
test_frame["rf_probability"] = rf_scores
test_frame["predicted_label"] = (test_frame["rf_probability"] >= 0.5).astype(int)

false_positives = test_frame[(test_frame["predicted_label"] == 1) & (test_frame["is_declining_label"] == 0)]
false_negatives = test_frame[(test_frame["predicted_label"] == 0) & (test_frame["is_declining_label"] == 1)]

print("False positives:", len(false_positives), " | False negatives:", len(false_negatives))

cols_to_show = ["content_id", "client_id", "rf_probability", "impressions_90d", "ctr", "avg_position", "trend_direction"]
false_positives.sort_values("rf_probability", ascending=False)[cols_to_show].head(5)

false_negatives.sort_values("rf_probability")[cols_to_show].head(5)

# --- Carry the Week-4 anomalies into the model's top-ranked output ---
top_50_by_model = test_frame.sort_values("rf_probability", ascending=False).head(50)

print("Rows with ctr == 0.0 in the model's top 50:", (top_50_by_model["ctr"] == 0.0).sum())
print("\nClient concentration in the model's top 50:")
print(top_50_by_model["client_id"].value_counts().head(5))
print("\nTop 10 feature importances:")
print(importance.head(10))
print("\nFalse positives:")
print(false_positives.head(5))
print("\nFalse negatives:")
print(false_negatives.head(5))

False positives: 505  | False negatives: 236
Rows with ctr == 0.0 in the model's top 50: 40

Client concentration in the model's top 50:
client_id
client_f74efabef1    50
Name: count, dtype: int64

Top 10 feature importances:
impressions_90d          0.102278
days_with_impressions    0.101628
avg_position             0.098129
log_impressions_90d      0.097848
content_age_days         0.088663
age_tier_365+            0.036793
char_count               0.035421
word_count               0.032536
scroll_rate              0.031756
ctr                      0.029893
dtype: float64

False positives:
               content_id          client_id  search_volume  competition  \
69   content_95d488a56079  client_f74efabef1            0.0         0.00   
147  content_bec800684271  client_f74efabef1           20.0         0.09   
168  content_d7cbd76b788d  client_f74efabef1           10.0         0.04   
198  content_c3e86d4031b6  client_f74efabef1           20.0         0.34   
251  content_7dff534d

## Errors and interpretation

**Top features:** The model relies most heavily on exposure and age signals —
`impressions_90d`, `days_with_impressions`, `avg_position`, `log_impressions_90d`, and
`content_age_days` together account for roughly 40% of total importance, while `ctr` itself
ranks only 10th (0.030). This is a somewhat unexpected result for a CTR/Engagement Opportunity
Scoring lane: the model has effectively learned a general "exposure and age predict decline"
signal rather than a CTR-specific one. No single feature dominates near 1.0 importance, so there
is no obvious sign of leakage in this feature set — but the mismatch between the lane's intent
(CTR/engagement) and what the model actually leans on is worth flagging as a limitation.

**False positives** (model flags decline risk, page is actually stable): All 5 inspected false
positives belong to the same client (`client_f74efabef1`), reinforcing the concentration issue
below. These pages have moderately-to-highly elevated `baseline_score` (0.22–0.71) and high
`visibility_score`/`log_impressions_90d` — meaning even the Week-4 hand rule saw them as
plausible candidates. The model assigns them probabilities of 0.52–0.68 (just above the 0.5
threshold), consistent with a client whose pages happen to look "exposed and aging" in ways the
model generally associates with decline, without that pattern actually holding for this client
in this period.

**False negatives** (model misses an actual decliner): These cases share a very different
profile — near-zero `search_volume`/`competition`, very low `log_impressions_90d` (~1–2, i.e.
only a handful of real impressions), several `feedly article` rows with `main_intent = unknown`,
and one row with `word_count = 0`. Probabilities stay low (0.20–0.45) despite a real decline,
because the model's top features (impressions/days-with-impressions) push probability down
whenever exposure is thin — regardless of whether a real trend exists underneath. This matches
the "low-volume boundary case" pattern already flagged during the Week-4 baseline review: the
smallest, least-tracked pages are the hardest for both the rule and the model to score reliably.

**Client concentration — a serious finding, not a footnote:** All 50 of the model's top-ranked
test rows belong to a single client (`client_f74efabef1`), and the false-positive sample above
confirms this is not incidental — this client's pages are systematically over-flagged. Combined
with the test split holding only 6 clients, this strongly suggests the model is partly learning
"this client's exposure/age profile looks risky" rather than a signal that generalizes cleanly
across clients. This does not necessarily invalidate the Precision@50 number, but it means the
number should currently be read as "the model ranks one client's pages first," not "the model
finds the best 50 pages across the whole content inventory." A stronger validation would need
more held-out clients per fold, or a dedicated check of whether `client_f74efabef1`'s features
are legitimately the most extreme in this test set or an artifact of the small fold size.

**ctr = 0.0 anomaly persists into the model output:** 40 of the model's top 50 rows (80%) have
`ctr = 0.0` — the same anomaly flagged in Week 4's baseline review has carried through into the
learned model, and is if anything more concentrated at the top of the ranked list now. Combined
with `ctr`'s low feature importance (10th place), this suggests the model is not using `ctr = 0`
as a direct decision driver so much as it correlates with the exposure/age features the model
does rely on — worth a follow-up check before trusting these specific top rows at face value.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.